# 13. Modelo original: DES-CPL

Los capítulos 8 a 12 mostraron que los mejores modelos del benchmark empatan: XGBoost y Random Forest no se distinguen por Nemenyi, ni con un presupuesto de búsqueda cinco veces mayor (sección 12.2). Este capítulo propone un modelo que no compite con ellos sino que los combina: **DES-CPL**, una selección dinámica de ensambles que decide, para cada paciente, cuánto confiar en cada modelo según la calidad de sus probabilidades en pacientes parecidos.

El capítulo documenta el proceso completo, incluidos los intentos que no funcionaron, porque juntos responden una pregunta que va más allá del modelo: **¿dónde está el techo de desempeño de este problema y por qué?**

| Sección | Contenido |
|---|---|
| 13.1-13.2 | La limitación que motiva el modelo y su formulación, con la relación con la literatura |
| 13.3 | Piloto en un fold externo de la formulación inicial |
| 13.4 | Diagnóstico: ¿se puede saber qué modelo acierta con cada paciente? |
| 13.5 | Versión final (poda y encogimiento) en los cinco folds, con ablación |
| 13.6 | Búsqueda de alternativas: otros modelos de la lista y otra información |
| 13.7 | Síntesis: el límite explicativo de los datos |

**Qué es exploratorio y qué es definitivo.** Las secciones 13.3, 13.4 y 13.6 son exploratorias: sirvieron para diseñar el modelo y entender el problema, y se reportan completas, incluidos los resultados desfavorables. Las secciones 13.5 y 13.8 son las definitivas y siguen el protocolo registrado antes de ejecutarlas (`experiments/protocolo.md`); solo ellas sustentan la condición de superioridad.

**Cómo se ejecuta.** Los cálculos costosos están en scripts que escriben en `results/` y continúan donde quedaron si se interrumpen; el capítulo solo lee sus resultados. Para regenerarlos, en la raíz del repositorio:

```
python experiments/des_cpl.py --folds 0 1 2 3 4     # modelos base y DES-CPL (~45 min)
python experiments/diagnostico_errores.py            # sección 13.4 (minutos)
python experiments/cribado_modelos.py                # sección 13.6.1 (~20 min)
python experiments/cribado_modelos.py --folds 0 1 2 3 4 --candidatos "Extra Trees"
python experiments/limite_informativo.py             # sección 13.6.3 (~15 min)
```

El código del modelo está en `src/des_cpl.py` y el del protocolo en `src/experimento_des_cpl.py`, con sus pruebas en `tests/`.

In [ ]:
from config import *

from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score

from src import configuracion as cfg
from src import experimento_des_cpl as exd
from src.estadistica import correccion_holm, prueba_nadeau_bengio

cfg.N_JOBS = 4

# Versión definitiva: modelos base con la configuración elegida por Random
# Search con 60 evaluaciones (capítulo 12) y DES-CPL con 60 de Optuna.
CARPETA = RESULTADOS / "des_cpl_60"
RUTA_EVALUACION = RESULTADOS / "des_cpl_evaluacion_60.csv"
# Exploratorio: piloto de la formulación inicial con la configuración del E2.
RUTA_PILOTO = RESULTADOS / "des_cpl_evaluacion_v1.csv"
RUTA_DIAGNOSTICO = RESULTADOS / "des_cpl_diagnostico_errores.csv"
RUTA_CRIBADO = RESULTADOS / "cribado_modelos_fold0.csv"
RUTA_CRIBADO_FOLDS = RESULTADOS / "cribado_modelos_folds.csv"
RUTA_LIMITE = RESULTADOS / "limite_informativo.csv"
RUTA_SOTA = RESULTADOS / "benchmarks_sota.csv"     # LightGBM y CatBoost
FOLDS = [0, 1, 2, 3, 4]
BASE = list(exd.MODELOS)


def leer(ruta):
    if not ruta.exists():
        print(f"Falta {ruta.name}: ejecuta su script (ver la introducción).")
        return pd.DataFrame()
    return pd.read_csv(ruta)


mejores_e2 = exd.mejores_combinaciones(
    pd.read_csv(RESULTADOS / "tabla_maestra_clasificacion.csv"))
mejores = exd.con_presupuesto(mejores_e2, RESULTADOS / "convergencia_evaluaciones.csv",
                              RESULTADOS / "convergencia_reajustes.csv", exd.PRESUPUESTO)
pd.DataFrame({"balanceo": mejores["balanceo"],
              "AUC-PR con presupuesto E2": mejores_e2["auc_pr_media"],
              f"AUC-PR con {exd.PRESUPUESTO}": mejores["auc_pr_media"]}).round(4)

## 13.1. La limitación: modelos que empatan en promedio pero no en cada paciente

Que dos modelos tengan el mismo AUC-PR medio no significa que acierten con los mismos pacientes. La tabla siguiente lo mide con las predicciones de prueba de los dos finalistas (capítulo 10): cuánto coinciden en el orden de riesgo, a cuántos pacientes señala uno y no el otro dentro del 15 % de mayor riesgo, y qué AUC-PR alcanzaría un *oráculo* que, para cada encuentro, eligiera el modelo que más se acerca a su etiqueta. El oráculo no es alcanzable, porque usa la etiqueta; es la cota superior de lo que podría ganar una combinación que eligiera bien el modelo de cada paciente.

In [ ]:
prueba = pd.read_csv(RESULTADOS / "predicciones_prueba.csv")
y_prueba = prueba.pop("y_real").to_numpy()
a, b = prueba["p_xgboost (adasyn)"].to_numpy(), prueba["p_random_forest (ninguno)"].to_numpy()

corte_a, corte_b = np.quantile(a, 0.85), np.quantile(b, 0.85)
solo_uno = ((a >= corte_a) != (b >= corte_b)).mean()
oraculo = np.where(np.abs(a - y_prueba) < np.abs(b - y_prueba), a, b)

complementariedad = pd.DataFrame({
    "indicador": ["Correlación de Spearman entre los dos modelos",
                  "Encuentros señalados por uno solo (15 % de mayor riesgo)",
                  "AUC-PR de XGBoost", "AUC-PR de Random Forest",
                  "AUC-PR del promedio de los dos",
                  "AUC-PR del oráculo (cota superior)"],
    "valor": [spearmanr(a, b).statistic, solo_uno,
              average_precision_score(y_prueba, a),
              average_precision_score(y_prueba, b),
              average_precision_score(y_prueba, (a + b) / 2),
              average_precision_score(y_prueba, oraculo)],
}).round(4)
guardar_resultado(complementariedad, "des_cpl_complementariedad")
complementariedad

Los dos modelos ordenan casi igual a los pacientes, pero no señalan exactamente a los mismos, y la distancia entre su AUC-PR y la del oráculo es grande. Promediar ya recupera una parte. La pregunta del modelo original es si una combinación que **dependa del paciente** recupera más que el promedio, que aplica la misma regla a todos.

## 13.2. Formulación

DES-CPL tiene cuatro componentes. La formulación completa, con sus dos proposiciones, está en el artículo; aquí se resume.

1. **Conjunto de selección fuera de pliegue.** Cada modelo base $f_m$ se reentrena en una validación cruzada de tres pliegues agrupada por paciente, y para cada encuentro se guarda la probabilidad del modelo que no lo vio, $\tilde f_m(x_i)$. Medir la competencia sobre los encuentros de entrenamiento haría parecer competentes a todos los modelos.
2. **Región de competencia estratificada y agrupada.** La posición de un encuentro es su perfil de salida, $\phi(x) = (\operatorname{logit} f_1(x), \dots, \operatorname{logit} f_M(x))$. La región reúne sus $k$ vecinos positivos y sus $k$ vecinos negativos más cercanos en ese espacio, excluyendo los encuentros del mismo paciente. Con un 11 % de positivos, una región sin estratificar casi nunca tendría positivos.
3. **Competencia por Brier local.** La competencia de cada modelo es su puntuación de Brier en la región, que mide la calidad de la probabilidad y no solo el acierto de la etiqueta:
$$
C_m(x) = \hat q(x)\,\frac{1}{k}\sum_{i \in \mathcal N_k^+(x)} \big(\tilde f_m(x_i) - 1\big)^2 + \big(1 - \hat q(x)\big)\,\frac{1}{k}\sum_{i \in \mathcal N_k^-(x)} \tilde f_m(x_i)^2 ,
$$
donde $\hat q(x)$ es la prevalencia local: la proporción de positivos entre los $2k$ vecinos de $x$ sin estratificar, suavizada hacia la prevalencia global.
4. **Ponderación softmax con temperatura.** $w_m(x) \propto \exp(-C_m(x)/\tau)$ y $\hat f(x) = \sum_m w_m(x) f_m(x)$. Con $\tau \to \infty$ es el promedio simple; con $\tau = 0$, la selección del modelo localmente mejor.

**Una corrección al diseño inicial.** La primera versión ponderaba cada clase con $1/2$ (Brier balanceado). Esa medida no es una regla de puntuación propia para el riesgo de la zona: en una región con la mitad de positivos, la probabilidad que minimiza el Brier balanceado es $0.5$ aunque el riesgo real de la zona sea del 5 %, de modo que premia al modelo cuya probabilidad está más cerca de $0.5$, que en este problema es casi siempre el de probabilidad más alta. Con la prevalencia local, la región sigue garantizando positivos (lo que reduce la varianza de la estimación en la clase minoritaria) y la reponderación devuelve a cada clase su peso real: es el estimador estratificado del Brier de la zona. El piloto (13.3) compara las dos versiones.


### 13.2.1. Relación con la literatura y qué es nuevo

La selección dinámica de ensambles no aparece en los artículos de readmisión en pacientes diabéticos de las búsquedas en Web of Science y Scopus (257 y 83 registros), ni en una búsqueda complementaria. El trabajo más cercano en readmisión es el de Shi et al. (2025), que aplica META-DES con regiones de competencia definidas por complicaciones a pacientes con cirrosis, y que reporta que los métodos de selección dinámica existentes no siempre superan a los ensambles del estado del arte en datos tabulares. Nótese que el conjunto «diabetes» que aparece en los bancos de prueba de selección dinámica (Pima) es de diagnóstico de diabetes, no de readmisión.

Cada componente de DES-CPL tiene antecedentes, y la novedad está en cómo se combinan y en el problema:

| Componente | Antecedente | Diferencia en DES-CPL |
|---|---|---|
| Región en el espacio de salidas | KNOP (Cavalcanti et al.) | Logit de las probabilidades, no votos |
| Competencia probabilística | RRC (Woloszynski y Kurzynski, 2011) | Mide la calidad de la probabilidad (Brier), no la probabilidad de acertar la etiqueta |
| Competencia por error | Selección dinámica de regresores (Rooney et al.) | Aplicada a probabilidades de clasificación con clases desbalanceadas |
| Región con ambas clases | KNORA-B/BI, DES-MI (García et al., 2018) | Además repondera cada clase con la prevalencia local estimada: estimador estratificado del Brier de la zona |
| Exclusión del paciente | — | Necesaria cuando un paciente tiene varios encuentros |
| Encogimiento local-global (13.5) | — | Mezcla la competencia local con la global con λ elegido en los datos; no lo encontramos en la literatura de selección dinámica |
| Poda (13.5) | Selección estática de clasificadores | Se decide en el conjunto de selección, antes de la combinación dinámica |

## 13.3. Piloto: la formulación inicial en un fold externo

**Objetivo.** Antes de los cinco folds, el piloto respondió tres preguntas con un costo de cómputo cinco veces menor: (1) si el protocolo reproduce los modelos base del Entregable 2, (2) cuál de las dos ponderaciones se adopta y (3) si la formulación inicial supera a los modelos base. Esta sección conserva sus resultados tal como se obtuvieron, antes de los cambios de la sección 13.5.

**Diseño.** Se usa el fold externo 0 del Entregable 2. Cada modelo base conserva su mejor combinación de balanceo y los hiperparámetros que eligió su bucle interno **en ese fold**, de modo que DES-CPL no recibe información que los modelos base no tuvieron. Se comparan:

| Familia | Métodos |
|---|---|
| Modelos base | Los siete, ajustados con todo el entrenamiento del fold |
| Ensambles estáticos | Promedio simple; apilamiento con regresión logística sobre los perfiles; pesos globales (la misma softmax con el Brier de cada modelo en todo el conjunto de selección, que aísla lo que aporta que la competencia sea local) |
| Modelo original | DES-CPL con ponderación por prevalencia local y con Brier balanceado |
| Ablación | Competencia por exactitud, región sin estratificar, sin excluir al paciente, selección dura ($\tau = 0$) |

Los hiperparámetros de cada método ($k \in \{5, 10, 20, 30, 50, 100\}$ y diez valores de $\tau$ entre $0$ e $\infty$: 60 configuraciones, el presupuesto validado en el capítulo 12) se eligen maximizando el AUC-PR **en el conjunto de selección**, sin tocar el fold de evaluación.

**Regla de decisión, fijada antes de ver el fold de evaluación.** La ponderación que se adopta es la de mayor AUC-PR en el conjunto de selección; el fold de evaluación solo se usa para reportar. Un fold no basta para afirmar superioridad: las pruebas de hipótesis se hacen con los cinco.

In [ ]:
# Las predicciones de los modelos base se calculan una vez por fold y se
# reutilizan en todo el capítulo (experiments/des_cpl.py hace lo mismo).
estado = exd.progreso(CARPETA, FOLDS)
if (estado["estado"] != "Completo").any():
    train = leer_tabla("diabetes_train")
    roles = roles_variables()
    PREDICTORES = (roles["numericas"] + roles["ordinales"]
                   + roles["binarias"] + roles["categoricas"])
    X, y = train[PREDICTORES], train[roles["objetivo"]]
    grupos = train[roles["identificador"]]
    particiones = exd.particiones_externas(X, y, grupos)
    for fold in estado.query("estado != 'Completo'")["fold"]:
        exd.ejecutar_fold(fold, particiones[fold], X, y, grupos, roles, mejores,
                          CARPETA, razon_desbalance=float((1 - y.mean()) / y.mean()))
    estado = exd.progreso(CARPETA, FOLDS)
estado

### 13.3.1. Control: reproducción de los modelos base

El modelo final de cada base es el mismo pipeline, con los mismos hiperparámetros y la misma semilla, ajustado con los mismos datos que en el estudio de convergencia del capítulo 12 (configuración elegida con 60 evaluaciones). Su AUC-PR en cada fold de evaluación debe coincidir con la de ese estudio; si no coincide, el protocolo no es comparable.

In [ ]:
resultados = exd.evaluar(CARPETA, FOLDS, RUTA_EVALUACION)
base = resultados.query("familia == 'Modelo base'").copy()
base["auc_pr_e2"] = [mejores.loc[m, "auc_pr_por_fold"][f]
                     for m, f in zip(base["método"], base["fold"])]
base["diferencia"] = base["auc_pr"] - base["auc_pr_e2"]
control = base.pivot_table(index="método", columns="fold", values="diferencia")
print(f"Diferencia máxima absoluta: {base['diferencia'].abs().max():.2e}")
control.map(lambda v: f"{v:+.1e}")

Las diferencias deben estar en el orden del redondeo numérico. En la versión exploratoria con los hiperparámetros del Entregable 2, seis de los siete modelos lo cumplieron y Naive Bayes difirió hasta en 0,001 según el fold; si la discrepancia reaparece aquí, se registra como menor, porque Naive Bayes no entra en la combinación final (sección 13.5).

### 13.3.2. Resultados del piloto (formulación inicial, fold 0)

In [ ]:
piloto = leer(RUTA_PILOTO)
if not piloto.empty:
    p0 = piloto.query("fold == 0").set_index("método")
    tabla_piloto = p0[["familia", "k", "tau", "auc_pr_seleccion", "auc_pr"]].copy()
    tabla_piloto["vs. mejor base"] = tabla_piloto["auc_pr"] - p0.query("familia == 'Modelo base'")["auc_pr"].max()
    display(tabla_piloto.sort_values("auc_pr", ascending=False).round(4))

### 13.3.3. Lectura

- **La corrección de la ponderación funciona.** Con la prevalencia local, DES-CPL supera a la versión con Brier balanceado; esta última queda al nivel del promedio simple, coherente con que premia la probabilidad más cercana a 0,5.
- **Pero DES-CPL no supera al mejor modelo base.** Random Forest y los ensambles globales (pesos globales y apilamiento) quedan por encima, y lo mismo ocurre en el conjunto de selección, donde se eligieron los hiperparámetros: no es un efecto de mirar el fold de evaluación.
- **Lo local resta.** Entre las variantes, mejora la que tiene la región más grande (sin estratificar, $2k = 200$) y empeora la selección dura. Cuanto más local la decisión, peor el resultado.

La pregunta siguiente es por qué. Hay dos explicaciones posibles: que la competencia local exista pero DES-CPL la estime mal, o que no exista. La sección 13.4 las separa.

## 13.4. Diagnóstico: ¿se puede saber qué modelo acierta con cada paciente?

Si la calidad relativa de los modelos cambia de un paciente a otro, la diferencia de pérdida entre dos modelos debería poder predecirse. Se mide directamente: un gradient boosting aprende, fuera de pliegue y agrupando por paciente, la diferencia de pérdida de Brier entre pares de modelos, primero desde el perfil de salida (el espacio de DES-CPL) y después desde el perfil más las 38 variables del paciente. Un $R^2$ cercano a cero significa que no hay nada que aprender. Con el mismo modelo se construye la «competencia aprendida»: pesos softmax con la pérdida predicha en lugar de la de los vecinos (`experiments/diagnostico_errores.py`, fold 0).

In [ ]:
diagnostico = leer(RUTA_DIAGNOSTICO)
if not diagnostico.empty:
    r2 = diagnostico[diagnostico["análisis"].str.startswith("R²")]
    display(r2.pivot_table(index="análisis", columns="espacio", values="valor_seleccion").round(4))
    aprendida = diagnostico.query("análisis == 'Competencia aprendida'")
    mejor_aprendida = aprendida.loc[aprendida.groupby("espacio")["valor_seleccion"].idxmax()]
    display(mejor_aprendida[["espacio", "tau", "valor_seleccion", "auc_pr_evaluacion"]].round(4))

**Lectura.** El $R^2$ fuera de pliegue no pasa de 0,013 en ningún par ni en ningún espacio: qué modelo se equivoca menos con un paciente no se puede anticipar, ni con las salidas de los modelos ni con las variables clínicas. La competencia aprendida, que usa un modelo flexible en lugar de vecinos, tampoco supera a Random Forest. La explicación es la segunda: **no existe heterogeneidad local aprovechable** en la calidad de los modelos. Los modelos sí se equivocan con pacientes distintos (sección 13.1), pero esas discrepancias son ruido, no estructura.

Esto tiene una consecuencia práctica para el diseño: la competencia local de DES-CPL estima con ruido una cantidad que es casi constante. La sección 13.5 incorpora esa conclusión al modelo en lugar de ignorarla.

## 13.5. Versión final: poda y encogimiento

El diagnóstico sugiere dos cambios, ambos decididos solo con el conjunto de selección:

5. **Encogimiento hacia la competencia global.** $\tilde C_m(x) = (1-\lambda)\,C_m(x) + \lambda\,\bar C_m$, donde $\bar C_m$ es el Brier del modelo en todo el conjunto de selección. Es el estimador de tipo James-Stein para una media local ruidosa: cuando hay poca señal local, conviene acercarla a la global. Con $\lambda = 0$ es la selección dinámica de la sección 13.2 y con $\lambda = 1$, una ponderación global. **Que los datos elijan $\lambda$ convierte el modelo en una prueba**: si eligen $\lambda \approx 1$, confirman que no hay heterogeneidad local que aprovechar.
6. **Poda del conjunto.** Solo se combinan los modelos con mayor AUC-PR fuera de pliegue (4 de 7, o todos, según elija la búsqueda). Los débiles (Naive Bayes, árbol, k-NN) añaden ruido a la región y a la mezcla.

**Búsqueda y equidad.** Los hiperparámetros ($k \in \{5, 10, 20, 30, 50, 100\}$, diez temperaturas entre 0 e $\infty$, $\lambda \in \{0; 0{,}25; 0{,}5; 0{,}75; 0{,}9; 1\}$ y la poda a 4 o a 7 modelos) se eligen con **optimización bayesiana (Optuna, TPE) con 60 evaluaciones** por fold, maximizando el AUC-PR en el conjunto de selección y sin tocar el fold de evaluación. Es el presupuesto validado en el capítulo 12, y los benchmarks con los que se compara reciben el mismo: cada modelo base usa, en cada fold, la configuración que eligió Random Search con 60 evaluaciones. Cuando $\lambda = 1$, $k$ no influye. La ablación quita cada componente por separado y conserva la búsqueda de los demás, con el mismo presupuesto.

**Iteraciones de diseño.** Antes de esta versión se evaluaron dos, ambas de forma exploratoria con los hiperparámetros del Entregable 2: la formulación inicial (sección 13.3; 0,2245 de AUC-PR media en los cinco folds, por debajo de XGBoost) y la versión con poda y encogimiento elegidos por rejilla (0,2287, por encima de XGBoost sin significancia, $p = 0{,}23$). La versión de esta sección es la definitiva, ejecutada según el protocolo registrado en `experiments/protocolo.md`.

### 13.5.1. Resultados en los cinco folds

In [ ]:
por_fold = resultados.pivot_table(index="método", columns="fold", values="auc_pr")
familias = resultados.drop_duplicates("método").set_index("método")["familia"]
# Benchmarks del estado del arte, con el mismo protocolo (experiments/benchmarks_sota.py).
sota = leer(RUTA_SOTA)
if not sota.empty:
    por_fold = pd.concat([por_fold, sota.pivot_table(index="modelo", columns="fold", values="auc_pr")])
    familias = pd.concat([familias, pd.Series("Estado del arte", index=sota["modelo"].unique())])
BENCHMARKS = BASE + (list(sota["modelo"].unique()) if not sota.empty else [])
# Mejor benchmark: mayor AUC-PR media del bucle externo (protocolo, sección 1).
REFERENCIA = por_fold.loc[BENCHMARKS].mean(axis=1).idxmax()
print(f"Mejor benchmark: {REFERENCIA} ({por_fold.loc[REFERENCIA].mean():.4f})")
resumen = pd.DataFrame({
    "familia": familias,
    "AUC-PR media": por_fold.mean(axis=1),
    "sd": por_fold.std(axis=1),
    f"dif. vs {REFERENCIA}": por_fold.mean(axis=1) - por_fold.loc[REFERENCIA].mean(),
    "folds a favor": (por_fold.gt(por_fold.loc[REFERENCIA], axis=1)).sum(axis=1),
})
guardar_resultado(resumen, "des_cpl_resumen_folds")
resumen.sort_values("AUC-PR media", ascending=False).round(4)

In [ ]:
# Prueba 2 del protocolo: t corregida de Nadeau-Bengio, unilateral (H1:
# DES-CPL mejor), de DES-CPL frente a cada benchmark, con Holm entre ellas.
from src.estadistica import d_cohen, delta_cliff, magnitud_cliff

pruebas = pd.DataFrame([{
    "benchmark": b, **prueba_nadeau_bengio(por_fold.loc["DES-CPL"] - por_fold.loc[b]),
    "delta de Cliff": delta_cliff(por_fold.loc["DES-CPL"], por_fold.loc[b]),
    "g de Hedges": d_cohen(por_fold.loc["DES-CPL"], por_fold.loc[b]),
} for b in BENCHMARKS])
pruebas["p (Holm)"] = correccion_holm(pruebas["p"].to_numpy())
pruebas["magnitud (Cliff)"] = pruebas["delta de Cliff"].map(magnitud_cliff)
guardar_resultado(pruebas, "des_cpl_nadeau_bengio")
print(f"Comparaciones: {len(pruebas)} (corrección de Holm sobre todas)")
pruebas.round(4)

In [ ]:
# Prueba 1 del protocolo (global): Friedman y Nemenyi sobre todos los
# benchmarks con DES-CPL incluido en el universo.
from scipy.stats import friedmanchisquare
from src.estadistica import diferencia_critica
import scikit_posthocs as sp

universo = por_fold.loc[BENCHMARKS + ["DES-CPL"]].T          # filas: folds
friedman = friedmanchisquare(*[universo[c] for c in universo.columns])
rangos = universo.rank(axis=1, ascending=False).mean().sort_values()
cd = diferencia_critica(universo.shape[1], universo.shape[0])
nemenyi = sp.posthoc_nemenyi_friedman(universo.to_numpy())
nemenyi.index = nemenyi.columns = universo.columns
print(f"Friedman: chi2 = {friedman.statistic:.2f}, p = {friedman.pvalue:.4g}; "
      f"diferencia crítica de Nemenyi = {cd:.2f} posiciones de rango")
pd.DataFrame({"rango medio": rangos,
              "p de Nemenyi frente a DES-CPL": nemenyi.loc[rangos.index, "DES-CPL"]}).round(4)

In [ ]:
# Qué eligió la búsqueda en cada fold: si λ ≈ 1, los datos apagan la parte local.
elegido = resultados.query("método == 'DES-CPL'")[
    ["fold", "k", "tau", "lambda", "modelos_combinados", "auc_pr_seleccion", "auc_pr"]]
elegido.round(4)

### 13.5.2. Lectura

> **Pendiente.** Se escribe con la ejecución definitiva (`python experiments/des_cpl.py`), una vez registrado el protocolo. En la versión exploratoria la búsqueda eligió $\lambda = 1$ o $0{,}9$ en todos los folds y el encogimiento fue el único componente con aporte medible.

## 13.6. Búsqueda de alternativas

Antes de cerrar con DES-CPL se probaron las otras dos salidas posibles: otro modelo de la lista de candidatos (13.6.1-13.6.2) u otra información (13.6.3).

### 13.6.1. Cribado de la lista de modelos no usados en diabetes

Cada familia de la lista se probó con una implementación de biblioteca (o una mínima propia, en Rotation Forest), con hiperparámetros razonables y sin búsqueda, en el mismo fold 0. Es una cota optimista: si ni la versión de biblioteca se acerca a Random Forest, una implementación propia tampoco lo hará. El intervalo es un bootstrap de la diferencia remuestreando pacientes.

In [ ]:
cribado = leer(RUTA_CRIBADO)
if not cribado.empty:
    display(cribado.sort_values("auc_pr", ascending=False)
            [["candidato", "implementacion", "auc_pr", "dif_vs_rf", "ic95_bajo", "ic95_alto", "segundos"]]
            .round(4))

### 13.6.2. El único candidato por encima: Extra Trees en los cinco folds

In [ ]:
cribado_folds = leer(RUTA_CRIBADO_FOLDS)
if not cribado_folds.empty:
    et = cribado_folds.query("candidato == 'Extra Trees'").sort_values("fold")
    et["dif. vs XGBoost"] = et["auc_pr"] - et["auc_pr_xgb"]
    display(et[["fold", "auc_pr", "auc_pr_rf", "auc_pr_xgb", "dif. vs XGBoost"]].round(4))
    print(prueba_nadeau_bengio(et["dif. vs XGBoost"]))

**Lectura.** Cinco familias quedan significativamente por debajo de Random Forest (el intervalo excluye el cero); Random Subspace y Rotation Forest empatan. Extra Trees, el único por encima en el fold 0, empata exactamente con XGBoost en la media de los cinco folds y gana en dos: su ventaja inicial era ruido. **Cambiar de familia de modelos no mueve el techo.**

### 13.6.3. ¿Y con otra información?

Si ni combinar ni cambiar de modelo ayuda, el límite debería estar en la información. Se probaron tres fuentes que el ETL no usa, cada una contra el mismo modelo de referencia (`experiments/limite_informativo.py`): los códigos ICD-9 completos con encogimiento jerárquico (capítulo → categoría → código), el historial del paciente dentro del conjunto (encuentros previos y readmisión previa; supone que `encounter_id` es cronológico) y la etiqueta ordinal (`NO` < `>30` < `<30`) en lugar de la binaria. Es un estudio aparte, con su propio protocolo (CSV crudo, 5 pliegues agrupados por paciente sobre el mismo 80 % de entrenamiento, un mismo HistGradientBoosting para todas las variantes), así que sus cifras se comparan entre sí y no con las de la sección 13.5.

In [ ]:
limite = leer(RUTA_LIMITE)
if not limite.empty:
    tabla_limite = limite.pivot_table(index="variante", columns="fold", values="auc_pr")
    ref = tabla_limite.loc["Referencia (variables del ETL)"]
    resumen_limite = pd.DataFrame({
        "AUC-PR media": tabla_limite.mean(axis=1),
        "dif. vs referencia": tabla_limite.mean(axis=1) - ref.mean(),
        "pliegues a favor": tabla_limite.gt(ref, axis=1).sum(axis=1),
        "p (Nadeau-Bengio)": [prueba_nadeau_bengio(tabla_limite.loc[v] - ref)["p"]
                              if v != ref.name else np.nan for v in tabla_limite.index],
    })
    guardar_resultado(resumen_limite, "limite_informativo_resumen")
    display(resumen_limite.sort_values("AUC-PR media", ascending=False).round(4))

**Lectura.** La etiqueta ordinal no ayuda: lo que predice un reingreso tardío no ordena bien a quienes reingresan pronto, y forzar un puntaje compartido resta. Lo que sí mueve el desempeño es la información nueva: los códigos de diagnóstico completos y el historial del paciente, y sobre todo su combinación, que es la única variante de todo el capítulo que mejora en todos los pliegues por encima del margen de equivalencia práctica (0,005). No se adoptó como modelo original por dos razones: el mecanismo (codificación jerárquica de categorías, Micci-Barreca, 2001; uso de la jerarquía ICD-9 en readmisión, Jiang et al., 2018) ya está publicado, y la ventaja vendría de dar a un modelo información que los benchmarks no tienen. Queda como la dirección más clara de trabajo futuro.

## 13.7. Síntesis: el límite explicativo de los datos

In [ ]:
filas = []
def fila(intento, enfoque, valor, referencia, nota):
    filas.append({"intento": intento, "enfoque": enfoque, "AUC-PR": valor,
                  "referencia": referencia, "diferencia": valor - referencia, "nota": nota})

if not resultados.empty:
    ref = por_fold.loc[REFERENCIA].mean()
    fila("DES-CPL inicial", "Combinar localmente los 7 modelos",
         por_fold.loc["DES-CPL inicial (sin poda ni encogimiento)"].mean(), ref, "5 folds vs XGBoost (60)")
    fila("DES-CPL final", "Poda + encogimiento local-global",
         por_fold.loc["DES-CPL"].mean(), ref, "5 folds vs XGBoost (60)")
    fila("Pesos globales", "Combinar globalmente",
         por_fold.loc["Pesos globales (Brier)"].mean(), ref, "5 folds vs XGBoost (60)")
if not diagnostico.empty:
    fila("Competencia aprendida", "Predecir qué modelo acierta",
         mejor_aprendida["auc_pr_evaluacion"].max(), 0.2535, "fold 0 vs Random Forest (E2)")
if not cribado_folds.empty:
    fila("Extra Trees", "Mejor familia de la lista", et["auc_pr"].mean(),
         et["auc_pr_xgb"].mean(), "5 folds vs XGBoost (E2)")
if not limite.empty:
    for v in ["Etiqueta ordinal: puntaje compartido + corrección", "+ ICD-9 jerárquico + historial"]:
        fila(v, "Otra información (estudio aparte)", resumen_limite.loc[v, "AUC-PR media"],
             tabla_limite.loc["Referencia (variables del ETL)"].mean(), "5 pliegues vs su referencia")
sintesis = pd.DataFrame(filas)
guardar_resultado(sintesis, "limite_explicativo_sintesis")
sintesis.round(4)

**Lectura.** Con las 38 variables del ETL, el desempeño tiene un techo cercano a 0,23 de AUC-PR que ningún enfoque rompe: ni buscar más hiperparámetros (capítulo 12), ni combinar los modelos de forma local o global, ni cambiar de familia. El diagnóstico de la sección 13.4 explica por qué la combinación local no puede funcionar: no hay estructura en qué modelo acierta con cada paciente. La única mejora consistente aparece al añadir información que esas variables no contienen. **El límite es de la información disponible, no del algoritmo.**

Para el modelo original, esto significa que DES-CPL cumple los criterios de novedad e implementación propia, y que su veredicto de superioridad depende de la evaluación final en el conjunto de prueba (sección 13.8), que se fija en `experiments/protocolo.md` antes de ejecutarse.

## 13.8. Evaluación final en el conjunto de prueba

> **Pendiente.** Se ejecuta una sola vez, con las tres pruebas de hipótesis fijadas de antemano en el protocolo.